# M11 Lab — Time Series Foundations

**Datasets:** `air_quality_daily.csv`, `sales_monthly.csv` &nbsp;|&nbsp; **Anchor:** McKinney Ch11 p.350–420

Some cells are marked `# [AI-OFF]` — those must be completed without Gemma's help. Log every Gemma interaction in `AI_USE.md`.

## L11.1 — Parse datetime safely `[McKinney Ch11 p.350–360]`

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)

df = pd.read_csv('air_quality_daily.csv',
                 parse_dates=['date'],
                 date_format='%Y-%m-%d')
df.head()

## L11.2 — DatetimeIndex slicing `[McKinney Ch11 p.360–370]`

In [ ]:
df = df.set_index('date').sort_index()

# TODO: replace the years/months below with values that exist in your dataset
print(df.loc['2024'].shape)
print(df.loc['2024-03':'2024-05'].shape)
print(df.loc['2024-03'].shape)

## L11.3 — Resampling `[McKinney Ch11 p.380–395]`

In [ ]:
monthly = df['pm25'].resample('MS').mean()
monthly.head()

*(your rationale here — replace this text)* Why `.mean()` and not `.sum()` / `.max()` for PM2.5?

## L11.4 — Rolling and expanding windows `[McKinney Ch11 p.395–410]`

In [ ]:
df['pm25_7d']     = df['pm25'].rolling(window=7,  min_periods=7).mean()
df['pm25_30d']    = df['pm25'].rolling(window=30, min_periods=20).mean()
df['pm25_cumavg'] = df['pm25'].expanding(min_periods=30).mean()

df[['pm25', 'pm25_7d', 'pm25_30d', 'pm25_cumavg']].tail(10)

## L11.5 — Time-zone reasoning &nbsp;⚠️ **[AI-OFF]** `[McKinney Ch11 p.370–380]`

Run the cell. Then in the markdown cell that follows, write **3–5 sentences in your own words** explaining: what happened, why, and how you would design ingestion to avoid this class of error.

In [ ]:
# [AI-OFF]
# Spring-forward in America/New_York 2024: 02:30 does not exist
try:
    ts = pd.Timestamp('2024-03-10 02:30').tz_localize('America/New_York')
    print('Localised:', ts)
except Exception as e:
    print(type(e).__name__, '-', e)

**[AI-OFF] DST analysis — write your own answer:**

*(3–5 sentences here)*

## L11.6 — Chronological train/test `[Expert + Géron Ch15]`

In [ ]:
# Manual cut by date
cutoff = '2024-01-01'   # TODO: justify your cutoff
train = df.loc[:cutoff]
test  = df.loc[cutoff:]
print('train:', train.shape, 'test:', test.shape)

# TimeSeriesSplit cross-validation
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
for i, (train_idx, test_idx) in enumerate(tscv.split(df), 1):
    print(f'fold {i}: train={len(train_idx)}, test={len(test_idx)}')

---
## Submission checklist

- [ ] All cells run top-to-bottom from a fresh kernel
- [ ] L11.5 [AI-OFF] cell completed with 3–5 sentences of your own DST reasoning
- [ ] Cutoff date in L11.6 is justified in a markdown cell
- [ ] No call to `train_test_split` with `shuffle=True` anywhere in this notebook
- [ ] `AI_USE.md` lists every Gemma prompt, response, and your accept/edit/reject decision